In [ ]:
# ── CELL 0: Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.datasets import make_blobs, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from collections import defaultdict

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
np.random.seed(42)
print('Ready.')

In [ ]:
# ── CELL 1: The Four Types — One Map ──────────────────────────────────────────
#
# There are 4 main paradigms in ML. The choice between them is NOT a technical
# preference — it is dictated by ONE thing: what data do you actually have?
#
#   ┌─────────────────────────────────────────────────────────────────┐
#   │  DO YOU HAVE LABELS (ground truth answers)?                     │
#   │                                                                 │
#   │  YES, all data labeled  → SUPERVISED LEARNING                  │
#   │  NO labels at all       → UNSUPERVISED LEARNING                │
#   │  A FEW labeled, rest no → SEMI-SUPERVISED LEARNING             │
#   │  Labels come from env   → REINFORCEMENT LEARNING               │
#   │  Labels generated from  → SELF-SUPERVISED LEARNING             │
#   │    the data itself           (used for pretraining LLMs)       │
#   └─────────────────────────────────────────────────────────────────┘
#
# In production, this choice is made before you write a single line of model
# code. It is a DATA problem first, not an algorithm problem.

paradigms = {
    'Supervised':      {'labels': 'All labeled',        'output': 'Prediction'},
    'Unsupervised':    {'labels': 'None',                'output': 'Structure/Groups'},
    'Semi-supervised': {'labels': 'Few labeled (<10%)',  'output': 'Prediction'},
    'Reinforcement':   {'labels': 'Reward from env',     'output': 'Policy/Action'},
    'Self-supervised': {'labels': 'Auto from data',      'output': 'Rich representations'},
}

df = pd.DataFrame(paradigms).T
print(df.to_string())

In [ ]:
# ── CELL 2: Supervised Learning ───────────────────────────────────────────────
#
# CONCEPT
# -------
# You have (X, y) pairs — input features and the correct answer.
# The model learns a function f(X) → y by minimizing error on labeled examples.
#
#   Regression:     y is a number    (predict price, predict demand)
#   Classification: y is a category  (spam/not-spam, fraud/not-fraud)
#
# PRODUCTION REALITY
# ------------------
# This is 80% of what gets deployed commercially. But here's what nobody tells
# you: THE BOTTLENECK IS NOT THE ALGORITHM. IT IS GETTING LABELS.
#
# How labels come from in production:
#
#   1. Historical outcomes  — e.g., past transactions marked as fraud by ops team
#   2. Human annotators     — labeling platforms like Scale AI, Labelbox
#   3. Implicit feedback    — user clicked = positive, no click = negative (search)
#   4. Programmatic labels  — rule-based heuristics to bootstrap labeling
#   5. Weak supervision     — noisy labels from multiple cheap sources
#
# Example: email spam filter
#   Old way: users mark as spam (implicit feedback label)
#   The model trains on those marked emails
#   But: if nobody marks spam, no labels — model stays dumb
#
# THE CONCEPT DRIFT PROBLEM (critical for production)
# ---------------------------------------------------
# Your model was trained on data from 6 months ago.
# The real world changes. Fraudsters change tactics. Markets shift.
# The distribution of new data no longer matches training data.
# → Model performance silently degrades in production.
# → You MUST monitor prediction distributions + outcomes continuously.
#
# Real example: a churn model trained in Jan degrades by July because
# customer behavior changed. Without monitoring, you don't know until
# the business team asks "why are we losing customers?"

# Quick demo — supervised classification
np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=2, n_informative=2,
                            n_redundant=0, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)
acc = accuracy_score(y_test, model.predict(X_test))

print(f'Supervised accuracy on test set: {acc:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: labeled training data
colors = ['steelblue', 'tomato']
for cls in [0, 1]:
    mask = y_train == cls
    axes[0].scatter(X_train[mask, 0], X_train[mask, 1],
                    c=colors[cls], s=20, alpha=0.6, label=f'Class {cls}')
axes[0].set_title('Supervised: All data is LABELED\n(each point has a correct answer)')
axes[0].legend()

# Plot 2: decision boundary
xx = np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 300)
yy = np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 300)
XX, YY = np.meshgrid(xx, yy)
Z = model.predict(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
axes[1].contourf(XX, YY, Z, alpha=0.2, cmap='RdBu')
for cls in [0, 1]:
    mask = y_test == cls
    axes[1].scatter(X_test[mask, 0], X_test[mask, 1],
                    c=colors[cls], s=30, edgecolors='k', linewidth=0.4,
                    label=f'Class {cls} (test)')
axes[1].set_title(f'Learned Decision Boundary | Test Accuracy = {acc:.3f}')
axes[1].legend()
plt.tight_layout()
plt.show()

# --- Concept drift simulation ---
print('\n--- Concept Drift Simulation ---')
print('Training data distribution: mean ~0')
# Simulate drift: test distribution shifts
X_drifted = X_test + np.array([3.0, 3.0])  # distribution shifted
acc_drifted = accuracy_score(y_test, model.predict(X_drifted))
print(f'Accuracy on original test:  {acc:.3f}')
print(f'Accuracy after data drift:  {acc_drifted:.3f}')
print('→ Same model, same labels. Different input distribution → accuracy collapses.')
print('→ In production: monitor input feature distributions daily.')

In [ ]:
# ── CELL 3: Unsupervised Learning ─────────────────────────────────────────────
#
# CONCEPT
# -------
# No labels. Find hidden structure in data.
# The model discovers patterns the human didn't define in advance.
#
#   Clustering:            group similar things together
#   Dimensionality Reduct: compress data to fewer dimensions
#   Anomaly Detection:     find the unusual outliers
#   Association Rules:     "customers who buy X also buy Y"
#
# PRODUCTION REALITY
# ------------------
# Unsupervised learning is EVERYWHERE in production but often invisible:
#
#   Fraud detection (first pass)
#     → cluster normal transactions, flag outliers as suspicious
#     → no labeled fraud data needed to get started
#
#   Customer segmentation
#     → cluster customers by behavior → personalized marketing per segment
#     → not "predict" but "understand"
#
#   Log anomaly detection (very relevant for you as a backend engineer)
#     → cluster normal log patterns, alert when something doesn't fit
#     → same pattern as Kubernetes/Azure Monitor watching metric distributions
#
#   Recommender systems (cold start)
#     → when no user history exists, cluster users by demographics
#     → serve content popular in their cluster
#
# KEY PRODUCTION GOTCHA
# ---------------------
# Unsupervised models have NO objective metric to optimize.
# You cannot measure accuracy. You measure:
#   - Silhouette score (cluster quality)
#   - Downstream business metric (did churn decrease after segmentation?)
# This makes them harder to evaluate and easier to ship broken.

# Demo — K-Means clustering on unlabeled data
np.random.seed(42)
X_raw, _ = make_blobs(n_samples=400, centers=4, cluster_std=1.2, random_state=42)

# All we have is X — no labels
km = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = km.fit_predict(X_raw)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw data — no labels
axes[0].scatter(X_raw[:, 0], X_raw[:, 1], c='steelblue', s=20, alpha=0.5)
axes[0].set_title('Unsupervised: Raw Data\n(NO labels — just points in space)')

# Right: discovered clusters
palette = ['steelblue', 'tomato', 'seagreen', 'orange']
for c in range(4):
    mask = cluster_labels == c
    axes[1].scatter(X_raw[mask, 0], X_raw[mask, 1],
                    c=palette[c], s=20, alpha=0.6, label=f'Cluster {c}')
axes[1].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                c='black', s=150, marker='X', zorder=5, label='Centroids')
axes[1].set_title('Clusters Discovered (no labels were used)')
axes[1].legend()
plt.tight_layout()
plt.show()

# Production-style: simulate anomaly detection
print('\n--- Production Pattern: Unsupervised Anomaly Detection ---')
np.random.seed(42)
normal_tx = np.random.normal([100, 50], [20, 10], (500, 2))  # normal transactions
anomalies  = np.array([[500, 200], [480, 190], [10, 5], [520, 210]])  # fraud-like
all_tx     = np.concatenate([normal_tx, anomalies])

km_tx = KMeans(n_clusters=1, random_state=42, n_init=10)
km_tx.fit(normal_tx)  # fit only on normal data
distances = np.linalg.norm(all_tx - km_tx.cluster_centers_[0], axis=1)
threshold = np.percentile(distances[:500], 99)  # 99th percentile of normal

flagged = np.where(distances > threshold)[0]
print(f'Total transactions:  {len(all_tx)}')
print(f'Anomaly threshold:   distance > {threshold:.1f}')
print(f'Flagged as anomalous: {len(flagged)} rows (indices {flagged})')
print('→ Last 4 are the injected fraud-like transactions → caught without any labels')

In [ ]:
# ── CELL 4: Semi-supervised Learning ──────────────────────────────────────────
#
# CONCEPT
# -------
# You have a SMALL set of labeled data and a LARGE set of unlabeled data.
# Use the structure in the unlabeled data to improve the supervised model.
#
# WHEN THIS HAPPENS IN PRODUCTION (very common)
# ---------------------------------------------
# - Medical imaging: 100 labeled scans (radiologist time is expensive),
#   100,000 unlabeled scans sitting in PACS systems
# - Document classification: 50 manually tagged support tickets,
#   500,000 untagged ones in Zendesk
# - Fraud: 200 confirmed fraud cases, 10M normal transactions
#
# TWO MAIN APPROACHES
# -------------------
# 1. Self-training (pseudo-labeling):
#    a) Train on small labeled set
#    b) Predict on unlabeled data with HIGH confidence
#    c) Add those high-confidence predictions as new labels
#    d) Retrain on expanded labeled set
#    → Cheap, easy to implement, surprisingly effective
#
# 2. Label Propagation:
#    Similar data points should have similar labels.
#    Propagate known labels to nearby unlabeled points in feature space.
#
# PRODUCTION PATTERN: Active Learning
# ------------------------------------
# Instead of labeling randomly, ask humans to label ONLY the most uncertain
# predictions. Think: model says "I'm 52% sure this is fraud" — THAT is the
# one to send to a human, not the 99% sure cases.
#
# In Azure, this is supported natively in Azure Machine Learning's labeling
# projects — it routes the hardest examples to human reviewers.

# Pseudo-labeling demo
np.random.seed(42)
X_all, y_all = make_classification(n_samples=1000, n_features=2, n_informative=2,
                                    n_redundant=0, random_state=42)

# Only 50 labeled, 800 unlabeled, 150 test
X_lab,  X_rest, y_lab,  y_rest = train_test_split(X_all, y_all, train_size=50,  random_state=42)
X_unlab, X_test, y_unlab, y_test = train_test_split(X_rest, y_rest, test_size=150, random_state=42)

# Baseline: supervised only on 50 labels
m_base = LogisticRegression(max_iter=1000)
m_base.fit(X_lab, y_lab)
acc_base = accuracy_score(y_test, m_base.predict(X_test))

# Semi-supervised: pseudo-labeling
probs      = m_base.predict_proba(X_unlab)
confidence = np.max(probs, axis=1)
high_conf  = confidence > 0.90  # only take high-confidence predictions

X_pseudo = np.vstack([X_lab, X_unlab[high_conf]])
y_pseudo = np.concatenate([y_lab, m_base.predict(X_unlab[high_conf])])

m_semi = LogisticRegression(max_iter=1000)
m_semi.fit(X_pseudo, y_pseudo)
acc_semi = accuracy_score(y_test, m_semi.predict(X_test))

print('Semi-supervised (pseudo-labeling) results:')
print(f'  Original labeled samples:          50')
print(f'  High-confidence pseudo-labels:     {high_conf.sum()}')
print(f'  Total samples for retraining:      {len(X_pseudo)}')
print(f'  Baseline accuracy (50 labels):     {acc_base:.3f}')
print(f'  Semi-supervised accuracy:          {acc_semi:.3f}')
print(f'  Improvement:                       +{(acc_semi - acc_base)*100:.1f}%')
print()
print('→ Free accuracy improvement using data you already have.')
print('→ Threshold matters: too low → noisy pseudo-labels → hurts performance.')
print('→ In production, start at 0.95+ confidence and tune down carefully.')

In [ ]:
# ── CELL 5: Reinforcement Learning ────────────────────────────────────────────
#
# CONCEPT
# -------
# An AGENT takes ACTIONS in an ENVIRONMENT.
# The environment returns a REWARD (positive) or PENALTY (negative).
# The agent learns a POLICY: which action to take in each STATE to maximize
# cumulative reward over time.
#
# Key terms:
#   Agent:       the ML model / decision-maker
#   Environment: the world it interacts with
#   State:       what the agent observes right now
#   Action:      what the agent decides to do
#   Reward:      scalar signal from environment after the action
#   Policy:      the learned rule: state → action
#
# WHERE RL IS USED IN PRODUCTION
# ------------------------------
#   Recommendation systems:
#     State   = user history, context
#     Action  = which item to recommend
#     Reward  = click, purchase, dwell time
#     Used by: Netflix, TikTok, YouTube
#
#   Trading / portfolio optimization:
#     State   = market prices, portfolio holdings
#     Action  = buy / hold / sell
#     Reward  = profit / risk-adjusted return
#
#   Robotics / drone control:
#     State   = sensor readings
#     Action  = motor speeds
#     Reward  = reaching target without crashing
#
#   Ad bidding (real-time bidding, RTB):
#     State   = user profile, ad slot context
#     Action  = bid price
#     Reward  = conversion profit - bid cost
#     Used by: Google Ads, Meta
#
# PRODUCTION CHALLENGES
# ---------------------
# RL is the hardest ML paradigm to productionize because:
#
#   1. You need a simulator or live environment to train.
#      Training in production means real users bear the cost of bad decisions.
#
#   2. Reward shaping is hard.
#      If your reward signal is wrong, the agent finds unexpected shortcuts.
#      (e.g., agent trained to maximize clicks learns to show clickbait)
#
#   3. Exploration vs exploitation tradeoff.
#      To learn, it must try new actions (explore) — but in production,
#      bad actions cost money. This tension is never fully resolved.
#
#   4. Distribution shift from simulator to real world (sim-to-real gap).
#
# → Most production RL is offline RL (learn from logged past decisions)
#   or multi-armed bandits (simpler RL for A/B testing replacement).

# Simple multi-armed bandit — the production-friendly RL primitive
# Think: "which version of this button gets more clicks?"
# Like A/B testing, but the system adapts in real time.

np.random.seed(42)

class EpsilonGreedyBandit:
    """Simplest RL-ish thing that actually ships in production.
    Used for: UI variants, email subject lines, recommendation strategies.
    """
    def __init__(self, n_arms, epsilon=0.1):
        self.n_arms    = n_arms
        self.epsilon   = epsilon  # exploration rate
        self.counts    = np.zeros(n_arms)  # times each arm pulled
        self.values    = np.zeros(n_arms)  # estimated reward per arm

    def select(self):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_arms)  # explore
        return np.argmax(self.values)  # exploit

    def update(self, arm, reward):
        self.counts[arm] += 1
        # incremental mean update
        self.values[arm] += (reward - self.values[arm]) / self.counts[arm]

# Simulate 3 content variants with different true click rates
true_click_rates = [0.10, 0.25, 0.40]  # variant 2 is best
bandit = EpsilonGreedyBandit(n_arms=3, epsilon=0.15)

history = defaultdict(list)
n_rounds = 2000

for t in range(n_rounds):
    arm    = bandit.select()
    reward = int(np.random.rand() < true_click_rates[arm])
    bandit.update(arm, reward)
    history['arm'].append(arm)
    history['reward'].append(reward)

# Results
arm_counts = np.bincount(history['arm'], minlength=3)
total_reward = sum(history['reward'])
random_reward = int(n_rounds * np.mean(true_click_rates))  # random baseline

print('Multi-Armed Bandit — 3 Content Variants:')
print(f'  True click rates:   Variant 0={true_click_rates[0]:.0%}  '
      f'Variant 1={true_click_rates[1]:.0%}  Variant 2={true_click_rates[2]:.0%}')
print(f'  Times each selected: {arm_counts}  (should favor Variant 2)')
print(f'  Estimated rewards:   {bandit.values.round(3)}')
print(f'  Total clicks earned: {total_reward}  (random baseline: {random_reward})')
print(f'  Lift over random:    +{(total_reward - random_reward) / random_reward * 100:.1f}%')
print()
print('→ This is Epsilon-Greedy bandit — ships in production at Netflix, Spotify.')
print('→ Replace A/B testing when you want to stop showing users the loser variant.')
print('→ Much simpler than full RL — start here before deep Q-Networks.')

# Convergence plot
window = 200
rewards_smooth = pd.Series(history['reward']).rolling(window).mean()
plt.figure(figsize=(10, 4))
plt.plot(rewards_smooth, color='steelblue', lw=2)
plt.axhline(max(true_click_rates), color='tomato', ls='--', label=f'Best arm ({max(true_click_rates):.0%})')
plt.axhline(np.mean(true_click_rates), color='gray', ls=':', label=f'Random ({np.mean(true_click_rates):.0%})')
plt.title('Bandit Converges Toward Best Variant Over Time')
plt.xlabel('Round'); plt.ylabel(f'Click Rate ({window}-round avg)')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 6: Self-supervised Learning ──────────────────────────────────────────
#
# CONCEPT
# -------
# Labels are generated AUTOMATICALLY from the structure of the data itself.
# No human annotation needed. Scales to internet-sized datasets.
#
# This is how GPT, BERT, CLIP, and every frontier LLM is pretrained.
#
# Examples of self-supervised tasks:
#
#   Next token prediction (GPT):
#     Input:  "The capital of France is"
#     Label:  "Paris"   ← generated from the NEXT WORD in the text
#     The entire internet is free training data this way.
#
#   Masked language modeling (BERT):
#     Input:  "The [MASK] is 42"
#     Label:  "answer"  ← randomly masked word from the original text
#
#   Contrastive learning (SimCLR, CLIP):
#     Two augmentations of the same image → should be similar
#     Image vs random other image → should be different
#     Learn: what visual features are invariant
#
# WHAT YOU GET
# ------------
# A PRETRAINED MODEL with rich general representations of the world.
# Then you fine-tune it for your specific task with a TINY labeled dataset.
#
# This is why 50 labeled examples can fine-tune a model to medical-grade
# accuracy — it already knows language from self-supervised pretraining.
#
# PRODUCTION REALITY
# ------------------
# YOU do not run self-supervised pretraining. That costs millions of dollars
# and requires thousands of GPUs.
#
# What you DO in production:
#   1. Download pretrained model (Hugging Face, Azure AI Foundry, OpenAI API)
#   2. Fine-tune on your domain-specific labeled data (100s to 1000s of examples)
#   3. Deploy that fine-tuned model
#
# The pretraining is done by OpenAI, Google, Meta, Mistral.
# You stand on their shoulders.
#
# ANALOGY (for you as a backend engineer):
# Self-supervised pretraining = building the OS kernel
# Fine-tuning on your task    = building your app on top of the OS
# You don't write the kernel. You write the app.

# Minimal demo: next-character prediction (toy GPT)
text = (
    "the quick brown fox jumps over the lazy dog "
    "the fox ran quickly across the field "
    "a quick brown dog ran over the hill "
)

# Self-supervised label generation: predict next character
pairs = [(text[i], text[i+1]) for i in range(len(text)-1)]
chars = sorted(set(text))
c2i   = {c: i for i, c in enumerate(chars)}

# Count bigram transitions
trans = np.zeros((len(chars), len(chars)))
for ctx, nxt in pairs:
    trans[c2i[ctx], c2i[nxt]] += 1

# Normalize to probabilities (this is the "model")
row_sums = trans.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
probs = trans / row_sums

# Generate text — sampling from learned distribution
def generate(seed='t', length=40):
    result = seed
    ch = seed
    for _ in range(length):
        if ch not in c2i: break
        row  = probs[c2i[ch]]
        if row.sum() == 0: break
        ch   = np.random.choice(chars, p=row)
        result += ch
    return result

print('Self-supervised: learned from raw text, no human labels')
print(f'Vocabulary size: {len(chars)} chars')
print(f'Training pairs:  {len(pairs)} (auto-generated from text itself)')
print()
print('Generated text samples (bigram character model):')
for seed in ['t', 'f', 'q']:
    print(f'  seed="{seed}" → "{generate(seed, 50)[:50]}"')

print()
print('This is the same PRINCIPLE as GPT — just scaled to:')
print('  - trillion tokens instead of ~200 chars')
print('  - transformer instead of bigram count matrix')
print('  - billions of parameters instead of 26×26')

In [ ]:
# ── CELL 7: Production Decision Framework ─────────────────────────────────────
#
# The question engineers get in a real project:
# "We have this data and this business problem — which ML type do we use?"
#
# Here is the decision process used by experienced ML engineers:
#
#   STEP 1: What does the business actually need?
#     → A prediction per request?     → Supervised (regression/classification)
#     → Understand/segment customers? → Unsupervised (clustering)
#     → Optimize sequential decisions?→ Reinforcement Learning
#     → Process text/images at scale? → Self-supervised pretrained model
#
#   STEP 2: What data do you actually have?
#     → Labeled (X, y)?               → Supervised
#     → Only X, no labels?            → Unsupervised or self-supervised
#     → A few labels, lots of X?      → Semi-supervised or fine-tuning
#     → Interactions with env/users?  → Reinforcement Learning
#
#   STEP 3: What are the production constraints?
#     → Latency < 10ms?               → Simpler supervised model (tree/linear)
#     → Need interpretability?        → Supervised with explainability (SHAP)
#     → Need to update in real time?  → Online learning or bandit
#     → Rare event detection?         → Unsupervised anomaly detection first
#
#   STEP 4: What is the cost of being wrong?
#     → False positive expensive?     → Tune threshold, add human review tier
#     → Missing a rare event costly?  → Optimize recall, not precision
#     → Need explanation for audit?   → Explainable models (Stage 8 covers this)

# Concrete examples from real production systems
examples = [
    {
        'Business Problem':   'Predict which customers will churn next month',
        'Data Available':     'CRM history with past churners labeled',
        'ML Type':            'Supervised Classification',
        'Algorithm':          'XGBoost / LightGBM',
        'Production Gotcha':  'Retrain monthly — churn behavior drifts seasonally',
    },
    {
        'Business Problem':   'Segment customers for personalized campaigns',
        'Data Available':     'Transaction history, no segment labels',
        'ML Type':            'Unsupervised Clustering',
        'Algorithm':          'K-Means or DBSCAN',
        'Production Gotcha':  'Clusters have no names — domain expert must interpret',
    },
    {
        'Business Problem':   'Detect payment fraud in real-time',
        'Data Available':     'Transactions, ~0.1% confirmed fraud labels',
        'ML Type':            'Semi-supervised or Anomaly Detection',
        'Algorithm':          'Isolation Forest + supervised layer',
        'Production Gotcha':  'Class imbalance — train on balanced, deploy with threshold',
    },
    {
        'Business Problem':   'Which push notification content drives app opens?',
        'Data Available':     'Historical sends + open rates + user context',
        'ML Type':            'Contextual Bandit (RL-lite)',
        'Algorithm':          'LinUCB or Thompson Sampling',
        'Production Gotcha':  'Reward must fire immediately; delayed reward is hard to attribute',
    },
    {
        'Business Problem':   'Classify Arabic customer support tickets',
        'Data Available':     '200 labeled tickets, 50K unlabeled',
        'ML Type':            'Fine-tuned pretrained model (Self-supervised base)',
        'Algorithm':          'BERT/AraBERT fine-tuned with pseudo-labels',
        'Production Gotcha':  'Validate on UAE Arabic dialect — MSA pretraining may not transfer',
    },
]

df_ex = pd.DataFrame(examples)
print(df_ex[['Business Problem', 'ML Type', 'Algorithm']].to_string(index=False))
print()
for ex in examples:
    print(f"GOTCHA ({ex['ML Type']}):")
    print(f"  {ex['Production Gotcha']}")

In [ ]:
# ── CELL 8: Online vs Batch Learning — Production Architecture ────────────────
#
# One more dimension not usually covered in textbooks but critical in production:
# HOW does the model update after deployment?
#
# BATCH LEARNING (most common at first)
# -------------------
# - Train on historical data offline
# - Deploy frozen model
# - Retrain periodically (weekly / monthly)
# - Simple, auditable, safe
# - Problem: model goes stale between retraining runs (concept drift)
#
# ONLINE LEARNING (streaming updates)
# ------------------------------------
# - Model updates its weights incrementally with each new sample
# - Always up to date
# - Much harder to operate:
#   → Poisoning attacks (bad data corrupts model fast)
#   → Harder to roll back
#   → Needs careful monitoring
# - Used in: ad click prediction, fraud detection, recommendation engines
#
# MICRO-BATCH / SCHEDULED RETRAINING (practical middle ground)
# ------------------------------------------------------------
# - Retrain on rolling window (last 30 days)
# - Trigger retrain when drift detected
# - A/B test new model vs old model before full rollout
# - This is what most production ML pipelines do
#
# YOU with your Kafka knowledge:
# Online learning = model as a Kafka consumer
# Each event is a training sample — model updates in real time
# Exactly how TikTok's recommendation model works

# Simulate concept drift + batch retraining lifecycle
np.random.seed(42)

def generate_data(n, mean_shift=0.0):
    X = np.random.randn(n, 2) + mean_shift
    y = (X[:, 0] + X[:, 1] > mean_shift).astype(int)
    return X, y

months = 6
results = []

# Train once at month 0
X_init, y_init = generate_data(500, mean_shift=0.0)
model_static = LogisticRegression()
model_static.fit(X_init, y_init)

for month in range(months):
    shift = month * 0.4  # drift increases over time
    X_m, y_m = generate_data(200, mean_shift=shift)

    # Static model — trained once, never retrained
    acc_static = accuracy_score(y_m, model_static.predict(X_m))

    # Retrained model — refreshed each month
    model_retrained = LogisticRegression()
    model_retrained.fit(X_m, y_m)  # train on fresh data
    acc_retrained = accuracy_score(y_m, model_retrained.predict(X_m))

    results.append({'month': month, 'static': acc_static, 'retrained': acc_retrained, 'drift': shift})

df_res = pd.DataFrame(results)
print('Concept Drift — Static vs Retrained Model:')
print(df_res.to_string(index=False))

plt.figure(figsize=(10, 4))
plt.plot(df_res['month'], df_res['static'],    'tomato',    lw=2, marker='o', label='Static (trained once)')
plt.plot(df_res['month'], df_res['retrained'], 'seagreen',  lw=2, marker='s', label='Retrained monthly')
plt.title('Why Retraining Matters: Concept Drift Over 6 Months')
plt.xlabel('Month'); plt.ylabel('Accuracy')
plt.ylim(0.5, 1.0); plt.legend()
plt.tight_layout(); plt.show()

print()
print('→ Static model degrades as distribution shifts.')
print('→ Scheduled retraining maintains performance.')
print('→ In production: trigger retrain on PSI (Population Stability Index) threshold.')

In [ ]:
# ── CELL 9: Practice ──────────────────────────────────────────────────────────
#
# SCENARIO: You're a backend engineer at a fintech startup in Dubai.
# The product team gives you 3 problems. Map each to the right ML type,
# identify the data requirements, and flag the production gotcha.

problems = [
    {
        'id': 1,
        'problem': 'We have 2 years of card transactions. 0.08% are labeled fraud by ops team. '
                   'Fraud team wants a real-time flag at point of sale.',
        'answer_type':    'Semi-supervised (few labels) + Unsupervised anomaly detection',
        'answer_why':     '0.08% label rate is tiny. Use unsupervised first (Isolation Forest), '
                          'then layer a supervised model on the confirmed labels.',
        'prod_gotcha':    'Model must return in <50ms at POS terminal. '
                          'Feature engineering must be stateless (no DB join at inference time).',
    },
    {
        'id': 2,
        'problem': 'We have 500K users, no behavioral labels, no segments defined. '
                   'Marketing wants to send personalized offers next week.',
        'answer_type':    'Unsupervised Clustering',
        'answer_why':     'No labels exist. Use K-Means on RFM features '
                          '(Recency, Frequency, Monetary). Then hand segments to marketing.',
        'prod_gotcha':    'Cluster labels are arbitrary numbers — need human naming. '
                          'Cluster count (K) requires business input, not just elbow method.',
    },
    {
        'id': 3,
        'problem': 'You have 300 Arabic customer support tickets manually labeled into '
                   '8 categories. 50K unlabeled tickets sit in the queue.',
        'answer_type':    'Fine-tuned self-supervised model (e.g., AraBERT)',
        'answer_why':     '300 labels with 8 classes. Classic few-shot + pretrained model. '
                          'AraBERT pretrained on Arabic Wikipedia covers the domain.',
        'prod_gotcha':    'AraBERT trained on MSA — UAE dialect slang may degrade it. '
                          'Validate on a held-out set of UAE dialect tickets specifically.',
    },
]

print('=' * 70)
for p in problems:
    print(f'\nPROBLEM {p["id"]}: {p["problem"]}')
    print(f'  ML Type:     {p["answer_type"]}')
    print(f'  Why:         {p["answer_why"]}')
    print(f'  Prod Gotcha: {p["prod_gotcha"]}')
    print('-' * 70)

print()
print('KEY TAKEAWAYS FROM THIS LESSON')
print('━' * 45)
takeaways = [
    '1. The type of ML is determined by your DATA, not your algorithm preference.',
    '2. Supervised is dominant — but labeling is the real engineering problem.',
    '3. Unsupervised is used for anomaly detection + segmentation in nearly every system.',
    '4. Semi-supervised (pseudo-labeling) is free accuracy when you have lots of unlabeled data.',
    '5. RL in production means bandits first — full RL is rare outside big tech.',
    '6. Self-supervised pretraining is done ONCE by big labs. You fine-tune.',
    '7. Concept drift is the silent killer of supervised models in production.',
    '8. Build scheduled retraining pipelines from day one.',
]
for t in takeaways:
    print(f'  {t}')